In [ ]:
!pip install scikit-learn pandas numpy joblib

In [3]:
import os

# This lists everything in /kaggle/input
# so you can see the exact folder and file name
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/aryashah2k/nfuqnidsv2-network-intrusion-detection-dataset/NF-UQ-NIDS-v2.csv


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score, make_scorer
import joblib
import json
import os
from collections import Counter

# ─────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────
DATA_PATH         = r"D:\FYP\Webapp\dataset\NF-UQ-NIDS-v2.csv"
OUTPUT_DIR        = r"D:\FYP\Webapp\output"
TARGET            = "Attack"
CHUNK_SIZE        = 500_000
MIN_CLASS_SAMPLES = 5
MIN_PER_CLASS     = 800
MAX_PER_CLASS     = 80_000
TOTAL_TARGET      = 1_200_000

# Tuning controls
TUNE_SAMPLE_SIZE  = 250_000   # use subset for hyperparameter search speed
N_ITER_SEARCH     = 12
CV_FOLDS          = 3

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────
# 1. DEFINE CANDIDATE FEATURES
# ─────────────────────────────────────────
CANDIDATE_FEATURES = [
    "IN_BYTES", "IN_PKTS", "OUT_BYTES", "OUT_PKTS",
    "FLOW_DURATION_MILLISECONDS", "DURATION_IN", "DURATION_OUT",
    "MIN_TTL", "MAX_TTL", "LONGEST_FLOW_PKT", "SHORTEST_FLOW_PKT",
    "MIN_IP_PKT_LEN", "MAX_IP_PKT_LEN",
    "SRC_TO_DST_SECOND_BYTES", "DST_TO_SRC_SECOND_BYTES",
    "RETRANSMITTED_IN_BYTES", "RETRANSMITTED_IN_PKTS",
    "RETRANSMITTED_OUT_BYTES", "RETRANSMITTED_OUT_PKTS",
    "SRC_TO_DST_AVG_THROUGHPUT", "DST_TO_SRC_AVG_THROUGHPUT",
    "NUM_PKTS_UP_TO_128_BYTES", "NUM_PKTS_128_TO_256_BYTES",
    "NUM_PKTS_256_TO_512_BYTES", "NUM_PKTS_512_TO_1024_BYTES",
    "NUM_PKTS_1024_TO_1514_BYTES", "TCP_WIN_MAX_IN", "TCP_WIN_MAX_OUT",
    "ICMP_TYPE", "ICMP_IPV4_TYPE", "DNS_QUERY_ID", "DNS_QUERY_TYPE",
    "DNS_TTL_ANSWER", "FTP_COMMAND_RET_CODE",
    "L4_SRC_PORT", "L4_DST_PORT", "PROTOCOL", "TCP_FLAGS",
    "CLIENT_TCP_FLAGS", "SERVER_TCP_FLAGS",
]

# ─────────────────────────────────────────
# 2. PEEK HEADERS → compute usecols
# ─────────────────────────────────────────
print("Reading column names...")
all_cols = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
FEATURES, seen = [], set()
for f in CANDIDATE_FEATURES:
    if f in all_cols and f not in seen:
        FEATURES.append(f)
        seen.add(f)

USECOLS = FEATURES + [TARGET]
print(f"Total columns in CSV : {len(all_cols)}")
print(f"Feature columns used : {len(FEATURES)}  →  loading {len(USECOLS)} of {len(all_cols)} columns")

# ─────────────────────────────────────────
# 3. PHASE 1 – fast class count
# ─────────────────────────────────────────
print("\nPhase 1: counting class distribution…")
total_counts = pd.Series(dtype=int)
for chunk in pd.read_csv(DATA_PATH, usecols=[TARGET], chunksize=CHUNK_SIZE):
    chunk[TARGET] = chunk[TARGET].replace({"Benign": "Normal"}).fillna("Unknown")
    total_counts  = total_counts.add(chunk[TARGET].value_counts(), fill_value=0)

total_counts = total_counts.astype(int).sort_values(ascending=False)
print(f"\nFull dataset rows : {total_counts.sum():,}")
print(total_counts.to_string())

# ─────────────────────────────────────────
# 4. ADAPTIVE PER-CLASS SAMPLE TARGETS
# ─────────────────────────────────────────
valid_classes = total_counts[total_counts >= MIN_CLASS_SAMPLES].index
valid_counts  = total_counts[valid_classes]

proportional     = (valid_counts / valid_counts.sum() * TOTAL_TARGET).round().astype(int)
per_class_target = (
    proportional
    .clip(lower=MIN_PER_CLASS, upper=MAX_PER_CLASS)
    .combine(valid_counts, min)
)
per_class_rate = (per_class_target / valid_counts).clip(upper=1.0)

print(f"\nAdaptive sample targets per class:")
summary = pd.DataFrame({
    "total_rows":     valid_counts,
    "target_samples": per_class_target,
    "rate_%":         (per_class_rate * 100).round(2),
}).sort_values("total_rows", ascending=False)
print(summary.to_string())
print(f"\nTotal target samples : {per_class_target.sum():,}")

# ─────────────────────────────────────────
# 5. PHASE 2 – chunked load with adaptive rates
# ─────────────────────────────────────────
print("\nPhase 2: sampling dataset in chunks…\n")
chunk_samples  = []
rows_processed = 0

for i, chunk in enumerate(pd.read_csv(
    DATA_PATH, usecols=USECOLS, chunksize=CHUNK_SIZE, low_memory=True
)):
    chunk[TARGET]   = chunk[TARGET].replace({"Benign": "Normal"}).fillna("Unknown")
    chunk[FEATURES] = chunk[FEATURES].fillna(0).replace([np.inf, -np.inf], 0)
    chunk = chunk[chunk[TARGET].isin(valid_classes)]
    if chunk.empty:
        continue

    sampled = chunk.groupby(TARGET, group_keys=False).apply(
        lambda x: x.sample(
            n=max(1, int(len(x) * per_class_rate.get(x.name, 0.01))),
            random_state=42
        )
    )
    chunk_samples.append(sampled)
    rows_processed += len(chunk)

    if (i + 1) % 10 == 0:
        so_far = sum(len(s) for s in chunk_samples)
        print(f"  → {rows_processed:>12,} rows processed | sample so far: {so_far:,}")

df = pd.concat(chunk_samples, ignore_index=True)
print(f"\nFinal sampled dataset: {df.shape}")
print("\nLabel distribution:")
print(df[TARGET].value_counts().to_string())

# ─────────────────────────────────────────
# 6. ENCODE LABELS
# ─────────────────────────────────────────
le        = LabelEncoder()
y_encoded = le.fit_transform(df[TARGET])

label_mapping = {int(i): str(lbl) for i, lbl in enumerate(le.classes_)}
print("\nLabel mapping:")
print(json.dumps(label_mapping, indent=2))

# ─────────────────────────────────────────
# 7. PREPARE X AND y
# ─────────────────────────────────────────
X = df[FEATURES].values.astype(np.float32)
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
y = y_encoded

# ─────────────────────────────────────────
# 8. SCALE
# ─────────────────────────────────────────
scaler = StandardScaler()
X = scaler.fit_transform(X)

# ─────────────────────────────────────────
# 9. TRAIN / TEST SPLIT
# ─────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTraining samples : {len(X_train):,}")
print(f"Test samples     : {len(X_test):,}")

# ─────────────────────────────────────────
# 10. SQRT-SCALED CLASS WEIGHTS
# ─────────────────────────────────────────
train_counts = Counter(y_train)
n_train      = len(y_train)
n_cls        = len(train_counts)
sqrt_weights = {cls: n_train / (n_cls * np.sqrt(count)) for cls, count in train_counts.items()}
mean_w       = np.mean(list(sqrt_weights.values()))
sqrt_weights = {cls: w / mean_w for cls, w in sqrt_weights.items()}

# ─────────────────────────────────────────
# 11. HYPERPARAMETER TUNING (macro-F1)
# ─────────────────────────────────────────
print("\nTuning Random Forest for macro-F1…")

# stratified subsample for faster tuning
if len(X_train) > TUNE_SAMPLE_SIZE:
    _, X_tune, _, y_tune = train_test_split(
        X_train, y_train,
        test_size=TUNE_SAMPLE_SIZE,
        random_state=42,
        stratify=y_train
    )
else:
    X_tune, y_tune = X_train, y_train

print(f"Tuning set size: {len(X_tune):,}")

base_rf = RandomForestClassifier(
    class_weight=sqrt_weights,
    random_state=42,
    n_jobs=-1,
    oob_score=False
)

param_dist = {
    "n_estimators": [200, 300],
    "max_depth": [14, 16, 18],
    "min_samples_split": [20, 40],
    "min_samples_leaf": [10, 20, 40],
    "max_features": ["sqrt"],
    "max_samples": [0.6, 0.75],
}

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=42)
macro_f1 = make_scorer(f1_score, average="macro")

search = RandomizedSearchCV(
    estimator=base_rf,
    param_distributions=param_dist,
    n_iter=N_ITER_SEARCH,
    scoring=macro_f1,
    cv=cv,
    random_state=42,
    n_jobs=-1,
    verbose=1,
    refit=False
)
search.fit(X_tune, y_tune)

best_params = search.best_params_
print("Best CV macro-F1:", round(search.best_score_, 4))
print("Best params:", best_params)

# ─────────────────────────────────────────
# 12. TRAIN FINAL MODEL (best params)
# ─────────────────────────────────────────
print("\nTraining final Random Forest…")
rf_model = RandomForestClassifier(
    **best_params,
    class_weight=sqrt_weights,
    random_state=42,
    n_jobs=-1,
    oob_score=True,
    verbose=1
)
rf_model.fit(X_train, y_train)

# ─────────────────────────────────────────
# 13. OVERFITTING DIAGNOSTIC
# ─────────────────────────────────────────
train_acc = rf_model.score(X_train, y_train)
test_acc  = rf_model.score(X_test,  y_test)
oob_acc   = rf_model.oob_score_

print(f"\n── Overfitting Check ──")
print(f"  Train accuracy : {train_acc:.4f}")
print(f"  OOB   accuracy : {oob_acc:.4f}   (should be close to test)")
print(f"  Test  accuracy : {test_acc:.4f}")
gap = train_acc - test_acc
print(f"  Train–Test gap : {gap:.4f}  {'⚠ possible overfit' if gap > 0.05 else '✓ OK'}")

# ─────────────────────────────────────────
# 14. EVALUATE
# ─────────────────────────────────────────
y_pred = rf_model.predict(X_test)

print("\n── Classification Report ──")
print(classification_report(y_test, y_pred, target_names=list(label_mapping.values())))

print("\n── Confusion Matrix ──")
cm_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=list(label_mapping.values()),
    columns=list(label_mapping.values())
)
print(cm_df)

# ─────────────────────────────────────────
# 15. FEATURE IMPORTANCE
# ─────────────────────────────────────────
importances = pd.DataFrame({
    "feature":    FEATURES,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

print("\n── Top 10 Most Important Features ──")
print(importances.head(10).to_string(index=False))

# ─────────────────────────────────────────
# 16. SAVE ALL OUTPUT FILES
# ─────────────────────────────────────────
joblib.dump(rf_model,  os.path.join(OUTPUT_DIR, "rf_anomaly_model.pkl")); print("\nModel saved.")
joblib.dump(scaler,    os.path.join(OUTPUT_DIR, "scaler.pkl"));           print("Scaler saved.")
joblib.dump(FEATURES,  os.path.join(OUTPUT_DIR, "feature_list.pkl"));     print("Feature list saved.")

with open(os.path.join(OUTPUT_DIR, "label_mapping.json"), "w") as f:
    json.dump(label_mapping, f, indent=2)
print("Label mapping saved.")

importances.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"), index=False)
print("Feature importance saved.")

print(f"\n── All files saved to {OUTPUT_DIR} ──")
print(os.listdir(OUTPUT_DIR))

# ─────────────────────────────────────────
# 17. INFERENCE FUNCTION
# ─────────────────────────────────────────
def predict_from_wazuh_log(parsed_log: dict) -> dict:
    features  = joblib.load(os.path.join(OUTPUT_DIR, "feature_list.pkl"))
    _scaler   = joblib.load(os.path.join(OUTPUT_DIR, "scaler.pkl"))
    model     = joblib.load(os.path.join(OUTPUT_DIR, "rf_anomaly_model.pkl"))
    with open(os.path.join(OUTPUT_DIR, "label_mapping.json")) as f:
        label_map = json.load(f)

    fv = np.array([parsed_log.get(feat, 0) for feat in features], dtype=np.float32).reshape(1, -1)
    fv = np.nan_to_num(fv, nan=0.0, posinf=0.0, neginf=0.0)
    fv = _scaler.transform(fv)
    prediction    = model.predict(fv)[0]
    probabilities = model.predict_proba(fv)[0]

    normal_index = next((k for k, v in label_map.items() if v == "Normal"), None)
    normal_prob  = float(probabilities[int(normal_index)]) if normal_index else None

    return {
        "prediction":         label_map[str(prediction)],
        "confidence":         round(float(max(probabilities)), 4),
        "normal_probability": round(normal_prob, 4) if normal_prob else None,
        "probabilities": {label_map[str(i)]: round(float(p), 4) for i, p in enumerate(probabilities)},
    }

# ── Test: SSH brute-force simulation ──
sample_log = {
    "IN_BYTES": 12000, "OUT_BYTES": 400, "IN_PKTS": 200, "OUT_PKTS": 15,
    "FLOW_DURATION_MILLISECONDS": 150, "PROTOCOL": 6,
    "TCP_FLAGS": 2, "L4_SRC_PORT": 54321, "L4_DST_PORT": 22,
    "MIN_TTL": 64, "MAX_TTL": 64,
}
print("\n── Sample Inference ──")
print(json.dumps(predict_from_wazuh_log(sample_log), indent=2))


Reading column names...
Total columns in CSV : 46
Feature columns used : 40  →  loading 41 of 46 columns

Phase 1: counting class distribution…

Full dataset rows : 44,141,731
Attack
Normal            14620157
DDoS              12631341
DoS               10383877
scanning           2196755
Reconnaissance     1530158
xss                1426045
password            670208
injection           398233
Bot                  83371
Brute Force          72105
Infilteration        67450
Exploits             18376
Fuzzers              12952
Backdoor             11095
Generic               9536
mitm                  4383
ransomware            2002
Theft                 1404
Analysis              1318
Shellcode              854
Worms                  110
Unknown                  1

Adaptive sample targets per class:
                total_rows  target_samples  rate_%
Attack                                            
Normal            14620157          100000    0.68
DDoS              12631341        

C:\Users\Hp\AppData\Local\Temp\ipykernel_43640\1608331386.py:114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_43640\1608331386.py:114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_43640\1608331386.py:114: FutureWarning: D

  →    5,000,000 rows processed | sample so far: 54,251


C:\Users\Hp\AppData\Local\Temp\ipykernel_43640\1608331386.py:114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_43640\1608331386.py:114: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sampled = chunk.groupby(TARGET, group_keys=False).apply(
C:\Users\Hp\AppData\Local\Temp\ipykernel_43640\1608331386.py:114: FutureWarning: D

  →   10,000,000 rows processed | sample so far: 108,491


...

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=-1)]: Done  26 tasks      | elapsed:    3.3s
[Parallel(n_jobs=-1)]: Done 176 tasks      | elapsed:   18.1s
[Parallel(n_jobs=-1)]: Done 300 out of 300 | elapsed:   30.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.6s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    3.7s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:    6.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.1s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.8s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:    1.4s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.1s



── Overfitting Check ──
  Train accuracy : 0.9565
  OOB   accuracy : 0.9537   (should be close to test)
  Test  accuracy : 0.9544
  Train–Test gap : 0.0021  ✓ OK


[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.8s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:    1.4s finished



── Classification Report ──
                precision    recall  f1-score   support

      Analysis       0.69      0.99      0.81        92
      Backdoor       0.97      0.91      0.94        91
           Bot       1.00      1.00      1.00       445
   Brute Force       1.00      0.98      0.99       383
          DDoS       0.99      0.99      0.99     19992
           DoS       0.98      0.98      0.98     19992
      Exploits       0.75      0.62      0.67        91
       Fuzzers       0.67      0.86      0.75        92
       Generic       0.92      0.74      0.82        91
 Infilteration       0.91      0.11      0.20       359
        Normal       0.97      0.92      0.95     19991
Reconnaissance       0.84      0.93      0.88      8310
     Shellcode       0.95      0.96      0.95        92
         Theft       0.72      0.99      0.83        90
         Worms       0.86      0.82      0.84        22
     injection       0.87      0.85      0.86      2157
          mitm    

[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:    0.0s finished
[Parallel(n_jobs=12)]: Using backend ThreadingBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done  26 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 176 tasks      | elapsed:    0.0s
[Parallel(n_jobs=12)]: Done 300 out of 300 | elapsed:    0.0s finished
